In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.gold_restaurant;

In [0]:
source_db = "workspace.silver_restaurant" # change with the actual silver layer name
target_db = "workspace.gold_restaurant"

#Creating the dimensions and fact tables

##The date dimension

In [0]:
from pyspark.sql.functions import col, to_date, date_format, year, month, dayofmonth, dayofweek, when, hour, minute
df_orders = spark.read.table(f"{source_db}.orders")
df_details = spark.read.table(f"{source_db}.order_details")
df_weather = spark.read.table(f"{source_db}.weather")

dates_orders = df_orders.select(to_date(col("timestamp")).alias("full_date"))
dates_weather = df_weather.select(to_date(col("timestamp")).alias("full_date"))

unique_dates = dates_orders.union(dates_weather).distinct().dropna()

dim_date = unique_dates.select(
    date_format(col("full_date"), "yyyyMMdd").cast("int").alias("date_id"),
    col("full_date"),
    year(col("full_date")).alias("year"),
    month(col("full_date")).alias("month_number"),
    dayofmonth(col("full_date")).alias("day"),
    date_format(col("full_date"), "MMMM").alias("month_name"),
    dayofweek(col("full_date")).alias("day_of_week"),
    date_format(col("full_date"), "EEEE").alias("day_name_of_week")
).withColumn(
    "is_weekend", 
    when(col("day_of_week").isin([1, 7]), True).otherwise(False) 
)

display(dim_date)


##The time dimension

In [0]:
time_orders = df_orders.select(
    date_format(col("timestamp"), "HHmm").cast("int").alias("time_id"), # we can remove the cast to int if we want to keep it more comprehensible
    date_format(col("timestamp"), "HH:mm").alias("time_label"),
    hour(col("timestamp")).alias("hour"),
    minute(col("timestamp")).alias("minute")
)
time_weather = df_weather.select(
    date_format(col("timestamp"), "HHmm").cast("int").alias("time_id"),
    date_format(col("timestamp"), "HH:mm").alias("time_label"),
    hour(col("timestamp")).alias("hour"),
    minute(col("timestamp")).alias("minute")
)
unique_times = time_orders.union(time_weather).distinct().dropna()

dim_time = unique_times.withColumn(
    "day_time",
    when((col("hour") >= 6) & (col("hour") < 12), "Morning")
    .when((col("hour") >= 12) & (col("hour") < 17), "Afternoon")
    .when((col("hour") >= 17) & (col("hour") < 21), "Evening")
    .otherwise("Night") 
).orderBy("time_id")

display(dim_time)

##The menu dimension

In [0]:
dim_menu = spark.read.table(f"{source_db}.menu")
dim_menu = dim_menu.drop("ingestion_timestamp")
dim_menu = dim_menu.drop("source_system")
dim_menu.display()

##The employees dimension

In [0]:
dim_employees = spark.read.table(f"{source_db}.employees")
dim_employees = dim_employees.drop("ingestion_timestamp")
dim_employees = dim_employees.drop("source_system")
dim_employees.display()

##The weather dimension

In [0]:
dim_weather = spark.read.table(f"{source_db}.weather")
dim_weather = dim_weather.drop("ingestion_timestamp")
dim_weather = dim_weather.drop("source_system")
dim_weather = dim_weather.drop("timestamp")
dim_weather.display()

##The sales fact 

In [0]:
df_orders = df_orders.drop("ingestion_timestamp")
df_orders = df_orders.drop("source_system")
df_orders = df_orders.withColumn("date_id", date_format(col("timestamp"), "yyyyMMdd").cast("int"))
df_orders = df_orders.withColumn("time_id", date_format(col("timestamp"), "HHmm").cast("int"))
df_orders = df_orders.drop("timestamp")

df_details = df_details.drop("ingestion_timestamp")
df_details = df_details.drop("source_system")

fact_sales = ( 
            df_details
            .join(df_orders, on = ["order_id"])
            .join(dim_menu, on = ["item_id"])
            .select(
                "order_id",
                "detail_id",
                "item_id",
                "server_id",
                "weather_id",
                "date_id",
                "time_id",
                "order_type",
                "qty",
                (col("qty") * col("price")).alias("total_amount")
            )

)
fact_sales.display()

#Saving the tables

In [0]:
dim_time.write \
    .mode("overwrite") \
    .saveAsTable(f"{target_db}.dim_time")
dim_date.write \
    .mode("overwrite") \
    .saveAsTable(f"{target_db}.dim_date")

dim_menu.write \
    .mode("overwrite") \
    .saveAsTable(f"{target_db}.dim_menu")

dim_employees.write \
    .mode("overwrite") \
    .saveAsTable(f"{target_db}.dim_emp")

dim_weather.write \
    .mode("overwrite") \
    .saveAsTable(f"{target_db}.dim_weather")

fact_sales.write \
    .mode("overwrite") \
    .saveAsTable(f"{target_db}.fact_sales")


In [0]:
fact_sales.write \
    .mode("overwrite") \
    .saveAsTable(f"{target_db}.fact_sales")

#Constraints

##Primary keys

In [0]:
%sql
ALTER TABLE workspace.gold_restaurant.dim_menu ALTER COLUMN item_id SET NOT NULL;
ALTER TABLE workspace.gold_restaurant.dim_emp ALTER COLUMN emp_id SET NOT NULL;
ALTER TABLE workspace.gold_restaurant.dim_weather ALTER COLUMN weather_id SET NOT NULL;
ALTER TABLE workspace.gold_restaurant.dim_date ALTER COLUMN date_id SET NOT NULL;
ALTER TABLE workspace.gold_restaurant.dim_time ALTER COLUMN time_id SET NOT NULL;
ALTER TABLE workspace.gold_restaurant.fact_sales ALTER COLUMN order_id SET NOT NULL;
ALTER TABLE workspace.gold_restaurant.fact_sales ALTER COLUMN detail_id SET NOT NULL;

In [0]:
%sql
-- Drop existing constraints to avoid DELTA_CONSTRAINT_ALREADY_EXISTS error
ALTER TABLE workspace.gold_restaurant.dim_menu DROP CONSTRAINT IF EXISTS dim_menu_pk CASCADE;
ALTER TABLE workspace.gold_restaurant.dim_emp DROP CONSTRAINT IF EXISTS dim_emp_pk CASCADE;
ALTER TABLE workspace.gold_restaurant.dim_weather DROP CONSTRAINT IF EXISTS dim_weather_pk CASCADE;
ALTER TABLE workspace.gold_restaurant.dim_date DROP CONSTRAINT IF EXISTS dim_date_pk CASCADE;
ALTER TABLE workspace.gold_restaurant.dim_time DROP CONSTRAINT IF EXISTS dim_time_pk CASCADE;

-- Re-apply constraints
ALTER TABLE workspace.gold_restaurant.dim_menu ADD CONSTRAINT dim_menu_pk PRIMARY KEY (item_id);
ALTER TABLE workspace.gold_restaurant.dim_emp ADD CONSTRAINT dim_emp_pk PRIMARY KEY (emp_id);
ALTER TABLE workspace.gold_restaurant.dim_weather ADD CONSTRAINT dim_weather_pk PRIMARY KEY (weather_id);
ALTER TABLE workspace.gold_restaurant.dim_date ADD CONSTRAINT dim_date_pk PRIMARY KEY (date_id);
ALTER TABLE workspace.gold_restaurant.dim_time ADD CONSTRAINT dim_time_pk PRIMARY KEY (time_id);

In [0]:
# Drop if exists then add
spark.sql("ALTER TABLE workspace.gold_restaurant.fact_sales DROP CONSTRAINT IF EXISTS fact_sales_pk")
spark.sql("ALTER TABLE workspace.gold_restaurant.fact_sales ADD CONSTRAINT fact_sales_pk PRIMARY KEY (order_id, detail_id)")

##Foreign keys

In [0]:
%sql
-- Cleanup Foreign Keys
ALTER TABLE workspace.gold_restaurant.fact_sales DROP CONSTRAINT IF EXISTS fk_sales_menu;
ALTER TABLE workspace.gold_restaurant.fact_sales DROP CONSTRAINT IF EXISTS fk_sales_emp;
ALTER TABLE workspace.gold_restaurant.fact_sales DROP CONSTRAINT IF EXISTS fk_sales_weather;
ALTER TABLE workspace.gold_restaurant.fact_sales DROP CONSTRAINT IF EXISTS fk_sales_date;
ALTER TABLE workspace.gold_restaurant.fact_sales DROP CONSTRAINT IF EXISTS fk_sales_time;

-- Apply Foreign Keys
ALTER TABLE workspace.gold_restaurant.fact_sales 
    ADD CONSTRAINT fk_sales_menu FOREIGN KEY (item_id) REFERENCES workspace.gold_restaurant.dim_menu(item_id);

ALTER TABLE workspace.gold_restaurant.fact_sales 
    ADD CONSTRAINT fk_sales_emp FOREIGN KEY (server_id) REFERENCES workspace.gold_restaurant.dim_emp(emp_id);

ALTER TABLE workspace.gold_restaurant.fact_sales 
    ADD CONSTRAINT fk_sales_weather FOREIGN KEY (weather_id) REFERENCES workspace.gold_restaurant.dim_weather(weather_id);

ALTER TABLE workspace.gold_restaurant.fact_sales 
    ADD CONSTRAINT fk_sales_date FOREIGN KEY (date_id) REFERENCES workspace.gold_restaurant.dim_date(date_id);

ALTER TABLE workspace.gold_restaurant.fact_sales 
    ADD CONSTRAINT fk_sales_time FOREIGN KEY (time_id) REFERENCES workspace.gold_restaurant.dim_time(time_id);